# TinyGPT: data, training, steering experiments E5 + E6
Settings: **GPU T4**, **Internet on**. Run all cells (~45 min). Download `/kaggle/working/tinygpt_results.zip` at the end.

In [ ]:
import os, subprocess
REPO = "https://github.com/arcyadav/tiny-gpt-and-qwen-steering-study.git"
os.chdir("/kaggle/working")
if not os.path.exists("project"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "project"], check=True)
os.chdir("/kaggle/working/project")
print(os.listdir())

In [ ]:
!pip install -q -U transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data: 500k TinyStories (~114M tokens) + our BPE tokenizer

In [ ]:
!python -m tinygpt.data --out data --limit 500000

## 2. Train TinyGPT (12.2M params, 80M tokens, checkpoint every 500 steps)

In [ ]:
!python train.py --data data --out results/tinygpt 2>&1 | grep -E 'params|val loss|done|step +[0-9]*000 '

## 3. Sanity check: what does it write?

In [ ]:
from steer.core import TinyGPTSteerer
m = TinyGPTSteerer("results/tinygpt/ckpt.pt", "data/tokenizer.json")
for p, t in zip(["Once upon a time, there was a little girl named Lily. One day, she", "Tom had a red ball. He"],
                m.generate(["Once upon a time, there was a little girl named Lily. One day, she", "Tom had a red ball. He"], max_new_tokens=80)):
    print(p, t, "
")

## 4. Steering: E5 (= E1 + E2 on TinyGPT) and E6 (over training checkpoints)

In [ ]:
!python -m steer.experiments --model tinygpt --run-dir results/tinygpt --tokenizer data/tokenizer.json --exp e1 e2 e6 --out results/tinygpt_steer 2>&1 | grep -v 'Loading weights'

## 5. Download (final model + tokenizer + results; the intermediate checkpoints are left out)

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/export/data", exist_ok=True)
shutil.copytree("results/tinygpt_steer", "/kaggle/working/export/results/tinygpt_steer", dirs_exist_ok=True)
os.makedirs("/kaggle/working/export/results/tinygpt", exist_ok=True)
for f in ["ckpt.pt", "summary.json", "log.json"]:
    shutil.copy(f"results/tinygpt/{f}", "/kaggle/working/export/results/tinygpt/")
for f in ["tokenizer.json", "meta.json"]:
    shutil.copy(f"data/{f}", "/kaggle/working/export/data/")
shutil.make_archive("/kaggle/working/tinygpt_results", "zip", "/kaggle/working/export")
print("download /kaggle/working/tinygpt_results.zip from the Output panel")